Notebook to calculate total cases, peak height, peak week, and $R_0$ for each model's predictions.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as st
import mosqlient as mosq
from epiweeks import Week
from aux_func import get_data
from analysis import * 
import matplotlib.pyplot as plt 
import seaborn as sns 
from mosqlient.prediction_optimize import get_df_pars_ls

In [2]:
code_to_state = {33: 'RJ', 32: 'ES', 41: 'PR', 23: 'CE', 21: 'MA',
 31: 'MG', 42: 'SC', 26: 'PE', 25: 'PB', 24: 'RN', 22: 'PI', 27: 'AL',
 28: 'SE', 35: 'SP', 43: 'RS', 15: 'PA', 16: 'AP', 14: 'RR',  11: 'RO',
 13: 'AM', 12: 'AC', 51: 'MT', 50: 'MS', 52: 'GO', 17: 'TO', 53: 'DF',
 29: 'BA'}

state_to_code = {value: key for key, value in code_to_state.items()}

geo_dengue = [2931350,2933307,2302503,3119401,
              3549805,3541406,1200401,1200203,
              1716109,4113700,4103701,4104808,
              5201405,5102637,5215231]

geo_chik = [2211001,2931350,3143302,3119401,
            1721000,1716109,4104808,4219507,
            5103403,5102637] 

In [3]:
challenge = 'dengue_state'
col_region = 'adm_1'

In [4]:
df_dengue_state  = get_data(challenge)
df_dengue_state.date = pd.to_datetime(df_dengue_state.date)

In [5]:
df_preds = pd.read_csv(f'./predictions/predictions_all_models_{challenge}.csv.gz', index_col = 'Unnamed: 0')
df_preds.date = pd.to_datetime(df_preds.date)

df_preds.model.unique()

<ArrowStringArray>
[                                       '3rd_imdc_isi_isi-dengue',
                                    '3rd_imdc_purdue_neuralearth',
                                          '3rd_imdc_fiocruz_mard',
                                               '3rd_imdc_bsc_ghr',
                                      '3rd_imdc_fgv_pattern-blue',
                                '3rd_imdc_lncc_lncc_arp26_dengue',
                                        '3rd_imdc_procc_bb_model',
                                         '3rd_imdc_unesp_recogna',
                                '3rd_imdc_ifgw_inframind-proteus',
                                                'DS-OKSTATE-2026',
                              '3rd_imdc_ceri_returnoftheforecast',
                                          '3rd_imdc_nus_nus-cerm',
                             '3rd_imdc_lncc_surge_model26_dengue',
                                     '3rd_imdc_pucrio_arbocaster',
                                           

In [6]:


df_preds = df_preds.loc[df_preds.model.isin([ '3rd_imdc_emap_epidematicos_prophet_fixed', 
                                                '3rd_imdc_emap_epidematicos_sarimax_fixed',
                                                '3rd_imdc_bsc_ghr'

                                             ])]
df_preds.head()

,date,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,adm_1,id,validation,wis,model
16511,2022-10-09,3.000,5.0,8.0,14.0,26.0,46.0,72.2,101.05,137.075,11,14274,1,52.28,3rd_imdc_bsc_ghr
16512,2022-10-16,5.000,7.0,10.0,18.0,31.0,52.0,81.0,116.15,154.025,11,14274,1,52.28,3rd_imdc_bsc_ghr
16513,2022-10-23,5.000,7.0,12.0,22.0,40.0,68.0,111.1,151.05,190.050,11,14274,1,52.28,3rd_imdc_bsc_ghr
16514,2022-10-30,6.000,9.0,16.0,27.0,48.0,83.0,134.1,176.20,222.025,11,14274,1,52.28,3rd_imdc_bsc_ghr
16515,2022-11-06,9.975,14.0,19.0,33.0,58.5,108.0,167.0,214.20,273.025,11,14274,1,52.28,3rd_imdc_bsc_ghr


In [7]:
df_preds.model.unique()

<ArrowStringArray>
[                        '3rd_imdc_bsc_ghr',
 '3rd_imdc_emap_epidematicos_sarimax_fixed',
 '3rd_imdc_emap_epidematicos_prophet_fixed']
Length: 3, dtype: str

In [8]:
df_preds.isnull().sum()

date          0
lower_95      0
lower_90      0
lower_80      0
lower_50      0
pred          0
upper_50      0
upper_80      0
upper_90      0
upper_95      0
adm_1         0
id            0
validation    0
wis           0
model         0
dtype: int64

In [9]:
df_dengue_state = df_dengue_state.merge(
        df_preds[["date", col_region, "validation"]].drop_duplicates(
                subset=["date", col_region, "validation"],
                keep="first"
            ), on=["date", col_region]
    )

df_dengue_state.head()

,date,adm_1,casos,validation
0,2022-10-09,11,99,1
1,2022-10-09,12,31,1
2,2022-10-09,13,250,1
3,2022-10-09,14,2,1
4,2022-10-09,15,60,1


In [10]:
def process_single_combination(df_preds, df_dengue_state,col_region, region, model, validation, n_paths=1000, n_samples=100, seed=0):
    """
    Processa a simulação e métricas para uma única combinação de modelo e validação.
    """
    rng = np.random.default_rng(seed)
    
    # 1. Filtragem e preparação de parâmetros
    df_preds_m = df_preds.loc[(df_preds.model == model) & (df_preds[col_region]== region)]
    #print(df_preds_m.shape)
    if df_preds_m.empty:
        return None  # Retorna None se não houver dados para essa combinação
        
    df_pars = get_df_pars_ls(df_preds_m)
    df_pars.loc[(df_pars.mu <= 0) | (df_pars.mu.isna()), 'mu'] = 0.1
    df_pars.loc[(df_pars.sigma <= 0) | (df_pars.sigma.isna()), 'sigma'] = 0.1

    # 2. Estimativa de Rho e Marginais
    df_rho_input = df_pars.loc[(df_pars['validation'] == validation - 1)].merge(
        df_dengue_state, on=['date', col_region, 'validation'], how='left'
    )
    rho = estimate_rho(df_rho_input)
    marginals_uf = build_marginals(df_pars, col_region, region, validation)

    # 3. Geração das trajetórias (paths)
    paths = np.vstack([
        sample_path(marginals_uf, rho, random_state=rng)
        for _ in range(n_paths)
    ])

    # 4. Cálculo das métricas baseadas em todas as trajetórias
    season_total = paths.sum(axis=1)
    season_peak = paths.max(axis=1)

    # 5. Amostragem para ajuste da curva de Richards
    indices_aleatorios = rng.choice(paths.shape[0], size=n_samples, replace=False)
    amostras_paths = paths[indices_aleatorios, :]

    r0_dist = []
    pico_dist = []
    t_ini_dist = []
    max_c_dist = []
    richards_curves = []

    for i, path in enumerate(amostras_paths):
        r0_i, pico_i, t_ini_i, max_c_i, richards = otim_single_path(path)
        r0_dist.append(r0_i)
        pico_dist.append(pico_i)
        t_ini_dist.append(t_ini_i)
        max_c_dist.append(max_c_i)

        df_curve = pd.DataFrame({
            "sample": i,
            "week": np.arange(len(richards)),
            "richards": richards
        })

        richards_curves.append(df_curve)

    richards_df = pd.concat(richards_curves, ignore_index=True)

    r0_dist = np.array(r0_dist)
    pico_dist = np.array(pico_dist)
    t_ini_dist = np.array(t_ini_dist)
    max_c_dist = np.array(max_c_dist)

    # 6. Consolidação dos resultados em um dicionário de intervalos
    metrics = {
        'model': model,
        'validation': validation,
        col_region: region
    }
    
    # Dicionário mapeando o nome da métrica para o seu array de distribuição
    distributions = {
        'season_total': season_total,
        'season_peak': season_peak,
        'r0_dist': r0_dist,
        'pico_dist': pico_dist, 
        't_ini_dist': t_ini_dist, 
        'max_c_dist': max_c_dist
    }
    
    # Calcula os quantis automaticamente para cada métrica
    for name, dist in distributions.items():
        lo, med, hi = np.quantile(dist, [0.025, 0.5, 0.975])
        metrics[f'{name}_p25'] = lo
        metrics[f'{name}_p50'] = med
        metrics[f'{name}_p975'] = hi


    # Salva os resultados: 
    table_suffix = f"{region}_{model}_{validation}_{challenge}".replace("-", "_")

    df_pars.to_csv(f'./samples/pars_{table_suffix}.csv.gz', index = False)

    df_paths = (
        pd.DataFrame(paths)
        .reset_index(names="path")
        .melt(
            id_vars="path",
            var_name="week",
            value_name="cases"
        )
    )

    df_paths.to_csv(
        f"./samples/paths_{table_suffix}.csv.gz",
        index=False
    )

    # curvas de Richards
    richards_df.to_csv(
        f"./samples/richards_{table_suffix}.csv.gz",
        index=False
    )

    return metrics

In [11]:
def run_pipeline(df_preds, df_dengue_state, col_region, region, models_list, validations_list):
    """
    Executa o loop por todos os modelos e validações e retorna um DataFrame consolidado.
    """
    results = []
    
    for model in models_list:
        for validation in validations_list:
            #print(f"Processando: Modelo={model} | Validation={validation}...")
            try:
                res = process_single_combination(df_preds, df_dengue_state, col_region, region, model, validation)
                if res is not None:
                    results.append(res)
            except Exception as e:
                print(f"Erro ao processar Modelo={model}, Validation={validation}: {e}, {col_region}: {region}")
                continue
                
    # Transforma a lista de dicionários em um DataFrame estruturado
    df_results = pd.DataFrame(results)
    return df_results

In [12]:
%%time
# Listas de modelos e validações que você quer rodar
lista_modelos = df_preds.model.unique()
lista_validations = [2,3,4]

regions = df_preds[col_region].unique()


df_all_adm = pd.DataFrame()

for region_alvo in regions: 

    # Executa o pipeline
    df_final_resultados = run_pipeline(df_preds, df_dengue_state, col_region, region_alvo, lista_modelos, lista_validations)

    # Salva os resultados em um arquivo
    df_all_adm = pd.concat([df_all_adm, df_final_resultados], ignore_index=True) 


df_all_adm.head()

CPU times: user 1h 1min 42s, sys: 564 ms, total: 1h 1min 43s
Wall time: 1h 1min 42s


,model,validation,adm_1,season_total_p25,season_total_p50,season_total_p975,season_peak_p25,season_peak_p50,season_peak_p975,r0_dist_p25,...,r0_dist_p975,pico_dist_p25,pico_dist_p50,pico_dist_p975,t_ini_dist_p25,t_ini_dist_p50,t_ini_dist_p975,max_c_dist_p25,max_c_dist_p50,max_c_dist_p975
0,3rd_imdc_bsc_ghr,2,11,3926.356157,15216.357029,59468.095876,236.235715,1063.708920,4767.802096,1.361793,...,1.807475,16.951367,21.386877,27.897317,1.0,1.0,5.00,165.719311,923.498798,3581.758919
1,3rd_imdc_bsc_ghr,3,11,3203.128866,13791.388719,55005.081033,189.891079,796.071583,3183.561219,1.565958,...,1.572685,18.576123,18.597311,18.635170,1.0,1.0,1.00,288.079852,704.673571,3339.196533
2,3rd_imdc_bsc_ghr,4,11,836.872654,5195.159384,26876.807457,39.473632,246.393039,1263.093333,1.463165,...,1.528246,18.018261,18.832136,20.059293,1.0,1.0,1.00,37.154056,209.002973,1168.885171
3,3rd_imdc_emap_epidematicos_sarimax_fixed,2,11,465.445198,3835.978796,32142.057218,33.759454,137.517591,1417.101022,1.211715,...,1.415991,19.438504,31.234950,35.000000,1.0,1.0,1.00,19.398182,113.775765,1379.973201
4,3rd_imdc_emap_epidematicos_sarimax_fixed,3,11,769.949156,3103.612184,14421.214459,44.453844,225.775321,1309.749877,1.095223,...,1.775232,18.333006,34.454719,35.000000,1.0,1.0,18.05,25.085783,126.892198,519.942338


In [13]:
df_old = pd.read_csv(f'predictions/new_metrics_{challenge}.csv.gz')

#df_old['model'] = '3rd' + df_old.model.astype(str)

df_old = df_old.loc[~df_old.model.isin(['3rd_imdc_emap_epidematicos_prophet', 
                                                '3rd_imdc_emap_epidematicos_sarimax',
                                                '3rd_imdc_emap_epidematicos_sarimax_muni',
                                                '3rd_imdc_bsc_ghr'])]
df_old.head()

,model,validation,state,season_total_p25,season_total_p50,season_total_p975,season_peak_p25,season_peak_p50,season_peak_p975,r0_dist_p25,...,r0_dist_p975,pico_dist_p25,pico_dist_p50,pico_dist_p975,t_ini_dist_p25,t_ini_dist_p50,t_ini_dist_p975,max_c_dist_p25,max_c_dist_p50,max_c_dist_p975
0,3rd_imdc_isi_isi-dengue,2,12,8155.220843,9666.404427,11588.776658,225.289613,295.884608,419.352327,1.255347,...,1.304268,29.934870,31.934181,35.000000,1.0,1.0,1.000,221.703746,271.983601,316.168189
1,3rd_imdc_isi_isi-dengue,3,12,5296.213880,6433.353223,7836.201734,148.828929,190.215776,248.274815,1.259569,...,1.288510,29.806261,32.044642,34.895799,1.0,1.0,1.000,141.684160,175.709042,209.287538
2,3rd_imdc_isi_isi-dengue,4,12,7235.298114,8421.714410,9572.018224,199.337666,244.680701,301.015333,1.259767,...,1.289039,30.060873,31.744612,34.252873,1.0,1.0,1.000,199.472020,222.530214,257.894208
3,3rd_imdc_purdue_neuralearth,2,12,2120.596418,6765.841749,28221.394693,144.663450,611.092154,4306.154039,1.109418,...,2.135155,7.888542,18.482081,33.183729,1.0,1.0,5.525,74.984244,386.932102,2199.990377
4,3rd_imdc_purdue_neuralearth,3,12,1518.752445,5766.735068,25714.256030,98.427024,478.604529,3005.435595,1.236723,...,2.050592,11.531725,19.721003,31.194712,1.0,1.0,5.525,54.121028,342.182133,1735.561044


In [14]:
df_old.model.unique()

<ArrowStringArray>
[                                       '3rd_imdc_isi_isi-dengue',
                                    '3rd_imdc_purdue_neuralearth',
                                          '3rd_imdc_fiocruz_mard',
                                      '3rd_imdc_fgv_pattern-blue',
                                '3rd_imdc_lncc_lncc_arp26_dengue',
                                        '3rd_imdc_procc_bb_model',
                                         '3rd_imdc_unesp_recogna',
                                '3rd_imdc_ifgw_inframind-proteus',
                                                'DS-OKSTATE-2026',
                              '3rd_imdc_ceri_returnoftheforecast',
                                          '3rd_imdc_nus_nus-cerm',
                             '3rd_imdc_lncc_surge_model26_dengue',
                                     '3rd_imdc_pucrio_arbocaster',
                                             '3rd_imdc_emap_lstm',
                                        '3r

In [15]:
df_end = pd.concat([df_old, df_all_adm], ignore_index=True)

df_end.head()

,model,validation,state,season_total_p25,season_total_p50,season_total_p975,season_peak_p25,season_peak_p50,season_peak_p975,r0_dist_p25,...,pico_dist_p25,pico_dist_p50,pico_dist_p975,t_ini_dist_p25,t_ini_dist_p50,t_ini_dist_p975,max_c_dist_p25,max_c_dist_p50,max_c_dist_p975,adm_1
0,3rd_imdc_isi_isi-dengue,2,12.0,8155.220843,9666.404427,11588.776658,225.289613,295.884608,419.352327,1.255347,...,29.934870,31.934181,35.000000,1.0,1.0,1.000,221.703746,271.983601,316.168189,NaN
1,3rd_imdc_isi_isi-dengue,3,12.0,5296.213880,6433.353223,7836.201734,148.828929,190.215776,248.274815,1.259569,...,29.806261,32.044642,34.895799,1.0,1.0,1.000,141.684160,175.709042,209.287538,NaN
2,3rd_imdc_isi_isi-dengue,4,12.0,7235.298114,8421.714410,9572.018224,199.337666,244.680701,301.015333,1.259767,...,30.060873,31.744612,34.252873,1.0,1.0,1.000,199.472020,222.530214,257.894208,NaN
3,3rd_imdc_purdue_neuralearth,2,12.0,2120.596418,6765.841749,28221.394693,144.663450,611.092154,4306.154039,1.109418,...,7.888542,18.482081,33.183729,1.0,1.0,5.525,74.984244,386.932102,2199.990377,NaN
4,3rd_imdc_purdue_neuralearth,3,12.0,1518.752445,5766.735068,25714.256030,98.427024,478.604529,3005.435595,1.236723,...,11.531725,19.721003,31.194712,1.0,1.0,5.525,54.121028,342.182133,1735.561044,NaN


In [16]:
df_end.model.unique()

<ArrowStringArray>
[                                       '3rd_imdc_isi_isi-dengue',
                                    '3rd_imdc_purdue_neuralearth',
                                          '3rd_imdc_fiocruz_mard',
                                      '3rd_imdc_fgv_pattern-blue',
                                '3rd_imdc_lncc_lncc_arp26_dengue',
                                        '3rd_imdc_procc_bb_model',
                                         '3rd_imdc_unesp_recogna',
                                '3rd_imdc_ifgw_inframind-proteus',
                                                'DS-OKSTATE-2026',
                              '3rd_imdc_ceri_returnoftheforecast',
                                          '3rd_imdc_nus_nus-cerm',
                             '3rd_imdc_lncc_surge_model26_dengue',
                                     '3rd_imdc_pucrio_arbocaster',
                                             '3rd_imdc_emap_lstm',
                                        '3r

In [19]:
df_all_adm.model.unique()

<ArrowStringArray>
[                        '3rd_imdc_bsc_ghr',
 '3rd_imdc_emap_epidematicos_sarimax_fixed',
 '3rd_imdc_emap_epidematicos_prophet_fixed']
Length: 3, dtype: str

In [20]:
df_end.to_csv(f'predictions/new_metrics_{challenge}_new.csv.gz', index = False)